In [1]:
print("Check_Python")

Check_Python


In [2]:
import os
from pathlib import Path
import tensorflow as tf
from tensorflow.keras.models import load_model
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from sklearn.metrics import classification_report, confusion_matrix

In [3]:
PROJECT_ROOT = Path("..") 
DATA_DIR = PROJECT_ROOT / "data" / "raw"

TRAIN_DIR = DATA_DIR / "train"
VAL_DIR   = DATA_DIR / "val"
TEST_DIR  = DATA_DIR / "test"

TRAIN_DIR, VAL_DIR, TEST_DIR

(WindowsPath('../data/raw/train'),
 WindowsPath('../data/raw/val'),
 WindowsPath('../data/raw/test'))

In [4]:
model = load_model("best_model_1.h5")

In [5]:
IMG_SIZE = (224, 224)
BATCH_SIZE = 32
COLOR_MODE = "grayscale"

In [6]:
test_datagen = ImageDataGenerator(
    rescale=1.0 / 255
)
test_generator = test_datagen.flow_from_directory(
    TEST_DIR,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    color_mode=COLOR_MODE,
    class_mode="binary",
    shuffle=False
)

Found 624 images belonging to 2 classes.


In [7]:
results = model.evaluate(test_generator)
print("Accuracy:", results[1])

20/20 ━━━━━━━━━━━━━━━━━━━━ 6s 213ms/step - accuracy: 0.8766 - loss: 0.3564 - precision: 0.8581 - recall: 0.9615
Accuracy: 0.8766025900840759


In [8]:
y_pred = model.predict(test_generator)
y_pred = (y_pred > 0.5).astype(int)
y_true = test_generator.classes

20/20 ━━━━━━━━━━━━━━━━━━━━ 5s 215ms/step


In [9]:
print(classification_report(y_true, y_pred))

              precision    recall  f1-score   support

           0       0.92      0.74      0.82       234
           1       0.86      0.96      0.91       390

    accuracy                           0.88       624
   macro avg       0.89      0.85      0.86       624
weighted avg       0.88      0.88      0.87       624



### Brief Conclusion

<b>Accuracy = 86% </b>→ overall model is okay

- Class 1 (disease) :
  - Recall = 0.96 (strong)
  - Matlab: almost sab patients detect ho rahe hain

- Class 0 (normal) :
  - Recall = 0.74 
  - Matlab: kaafi normal log ko galat disease bol raha ha

### Use 3-Fold?

- Prevents overfitting on one dataset split
- Ensures model sees different pneumonia samples
- Improves generalization
- Gives more reliable accuracy

Instead of trusting 1 model → we trust average of 3 models

In [10]:
from tensorflow.keras.models import load_model
import numpy as np

model_fold1 = load_model("best_model_2_fold1.h5")
model_fold2 = load_model("best_model_2_fold2.h5")
model_fold3 = load_model("best_model_2_fold3.h5")

In [11]:
results1 = model_fold1.evaluate(test_generator)
results2 = model_fold2.evaluate(test_generator)
results3 = model_fold3.evaluate(test_generator)

20/20 ━━━━━━━━━━━━━━━━━━━━ 5s 209ms/step - accuracy: 0.8365 - loss: 0.4428 - precision: 0.8186 - recall: 0.9487
20/20 ━━━━━━━━━━━━━━━━━━━━ 5s 212ms/step - accuracy: 0.8830 - loss: 0.3028 - precision: 0.8661 - recall: 0.9615
20/20 ━━━━━━━━━━━━━━━━━━━━ 6s 224ms/step - accuracy: 0.8590 - loss: 0.4124 - precision: 0.8326 - recall: 0.9692


Ensemble evaluation

In [12]:
y_pred_avg = (
    model_fold1.predict(test_generator)+
    model_fold2.predict(test_generator)+
    model_fold3.predict(test_generator)
)/3
y_pred_avg = (y_pred_avg > 0.5).astype(int)
y_true = test_generator.classes

20/20 ━━━━━━━━━━━━━━━━━━━━ 5s 221ms/step
20/20 ━━━━━━━━━━━━━━━━━━━━ 5s 228ms/step
20/20 ━━━━━━━━━━━━━━━━━━━━ 5s 220ms/step


In [13]:
print("Avg Accuracy:", (results1[1]+results2[1]+results3[1])/3)

Avg Accuracy: 0.8595085342725118


In [14]:
print(classification_report(y_true, y_pred_avg))

              precision    recall  f1-score   support

           0       0.91      0.71      0.80       234
           1       0.84      0.96      0.90       390

    accuracy                           0.86       624
   macro avg       0.88      0.83      0.85       624
weighted avg       0.87      0.86      0.86       624



In [16]:
print(confusion_matrix(y_true, y_pred_avg))

[[165  69]
 [ 16 374]]


In [17]:
print(confusion_matrix(y_true, y_pred))

[[172  62]
 [ 15 375]]


<b>Accuracy = 86% </b>→ overall model is okay

- Class 1 (disease) :
  - Recall = 0.96 (still strong)
  - Matlab: almost sab patients detect ho rahe hain

- Class 0 (normal) :
  - Recall = 0.71 (3 fold didnt improve it) 
  - Matlab: kaafi normal log ko galat disease bol raha ha